In [233]:
%pip install -q dspy-ai requests==2.32.4 pandas==2.2.2 pypdf scikit-learn

In [234]:
import os
import dspy
from google.colab import userdata

# ---- USER FILLS THESE IN ----
AZURE_API_KEY = userdata.get('Azure')          # <-- put your key in Colab secrets or env var
AZURE_API_BASE = "https://o3miniapi.cognitiveservices.azure.com/"  # your endpoint
AZURE_API_VERSION = "2024-12-01-preview"
AZURE_DEPLOYMENT = "gpt-5-mini"       # <-- your chat-capable deployment name (e.g., "gpt-4o-mini")

# Recommended: use env var rather than hardcoding.
if AZURE_API_KEY:
    os.environ["AZURE_API_KEY"] = AZURE_API_KEY

# DSPy LM via LiteLLM Azure provider string.
# Note: provider strings can vary across LiteLLM versions; "azure/<deployment>" is the common pattern.
lm = dspy.LM(
    f"azure/{AZURE_DEPLOYMENT}",
    api_key=os.getenv("AZURE_API_KEY", ""),
    api_base=AZURE_API_BASE,
    api_version=AZURE_API_VERSION,
    model_type="chat",
    temperature=1.0,
    max_tokens=32000,
)

dspy.configure(lm=lm)


In [235]:
import requests
from typing import Dict, Any, List, Tuple

def get_dataset_payload(dataset_id: str) -> Dict[str, Any]:
    url = f"https://www.data.gouv.fr/api/1/datasets/{dataset_id}/"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

def extract_resource_urls(ds: Dict[str, Any]) -> List[str]:
    urls = []
    for res in (ds.get("resources") or []):
        u = res.get("url")
        if u:
            urls.append(u)
#    for res in (ds.get("community_resources") or []):
#        u = res.get("url")
#        if u:
#            urls.append(u)
    # de-dupe preserving order
    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def summarize_dataset_metadata(ds: Dict[str, Any]) -> str:
    org = ds.get("organization") or {}
    lines = [
        f"ID: {ds.get('id','')}",
        f"Title: {ds.get('title','')}",
        f"Description_short: {ds.get('description_short','')}",
        f"Description: {ds.get('description','')}",
        f"Organization: {org.get('name','')} (slug={org.get('slug','')}, id={org.get('id','')})",
        f"License: {ds.get('license','')}",
        f"Frequency: {ds.get('frequency','')}",
        f"Created_at: {ds.get('created_at','')}",
        f"Last_update: {ds.get('last_update','')}",
        f"Tags: {', '.join(ds.get('tags') or [])}",
        f"URI: {ds.get('uri','')}",
        f"Page: {ds.get('page','')}",
    ]
    return "\n".join([x for x in lines if x.strip()])


In [236]:
import os, re, json
import pandas as pd
from pypdf import PdfReader

def _safe_filename(url: str) -> str:
    name = re.sub(r"[^a-zA-Z0-9._-]+", "_", url.split("/")[-1].split("?")[0])
    return name or "resource"

def download_file(url: str, out_dir: str) -> str:
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, _safe_filename(url))
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return path

def extract_text_from_file(path: str, max_rows: int = 200) -> str:
    lower = path.lower()

    # CSV
    if lower.endswith(".csv"):
        df = pd.read_csv(path, nrows=max_rows)
        return f"[CSV preview: first {len(df)} rows]\n" + df.to_csv(index=False)

    # JSON
    if lower.endswith(".json") or lower.endswith(".jsonl"):
        if lower.endswith(".jsonl"):
            # take first N lines
            lines = []
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                for i, line in enumerate(f):
                    if i >= max_rows:
                        break
                    lines.append(line.strip())
            return "[JSONL preview]\n" + "\n".join(lines)
        else:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                obj = json.load(f)
            txt = json.dumps(obj, ensure_ascii=False, indent=2)
            # keep it bounded
            return txt[:200_000]

    # TXT / MD
    if lower.endswith(".txt") or lower.endswith(".md"):
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()[:200_000]

    # PDF
    if lower.endswith(".pdf"):
        reader = PdfReader(path)
        pages = []
        for i, p in enumerate(reader.pages[:20]):  # bound pages
            pages.append(p.extract_text() or "")
        return "[PDF extracted text]\n" + "\n\n".join(pages)[:200_000]

    # TEMPORARY fix for zip files - remove altogether
    if lower.endswith(".zip"):
        return f"[Unsupported binary format for text extraction: {os.path.basename(path)}]"

    # Fallback: try decode as text
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()[:200_000]
    except Exception:
        return f"[Unsupported binary format for text extraction: {os.path.basename(path)}]"


In [237]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def chunk_text(text: str, chunk_size: int = 2400, overlap: int = 200) -> List[str]:
    text = text.replace("\r", "\n")
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i+chunk_size])
        i += max(1, chunk_size - overlap)
    return chunks

class LocalTfidfRetriever:
    def __init__(self, documents: List[str]):
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.docs = documents
        self.mat = self.vectorizer.fit_transform(documents)

    def topk(self, query: str, k: int = 6) -> List[Tuple[int, float, str]]:
        q = self.vectorizer.transform([query])
        sims = cosine_similarity(q, self.mat)[0]
        idxs = sims.argsort()[::-1][:k]
        return [(int(i), float(sims[i]), self.docs[int(i)]) for i in idxs]


In [241]:
class GenerateQuestions(dspy.Signature):
    """Generate realistic user questions that AGORA could answer using this dataset and its resources."""
    dataset_metadata: str = dspy.InputField(desc="Metadata about the dataset: title, description, org, license, etc.")
    extracted_text_previews: str = dspy.InputField(desc="Short previews of extracted dataset resources, showing only the first N characters of each document.")
    # vvvv changed "top relevant" to "random"
    #retrieved_context: str = dspy.InputField(desc="Top relevant excerpts from the dataset resources (sample rows/text).")
    retrieved_context: str = dspy.InputField(desc="Random excerpts from the dataset resources (sample rows/text).")
    project_description: str = dspy.InputField(desc="AGORA project description & typical workflow constraints.")
    # #      had to modify this a little bit after noticing that the LLM returns a lot of questions which include
    # # VVVV specific data field names (eg. prix-max, prix-min, train-no, etc.), which is unrealistic
    # questions: str = dspy.OutputField(desc="A numbered list of 15-30 user questions. Each should be actionable, data-driven, and realistic. Remember that the user usually does not have any concrete information about the data, such as field names, or even the dataset as a whole. Only generate questiont that a user might ask which might PERTAIN to this dataset, assuming the user has no knowledge of the dataset's recoures or it's content, or even it's name or any information about it.")
    # ^^^^ DELETEED this because it just wasn't cutting it. I felt like the answers were still very technical. So I have this now:
    questions: str = dspy.OutputField(
    desc=(
        "A numbered list of 15–30 realistic end-user questions, "
        "Each question must be phrased in natural, non-technical language and reflect "
        "real decision-making or feasibility analysis (e.g., location suitability, demand, "
        "infrastructure, regulation, competition, or trends).\n\n"

        "The user has NO knowledge of:\n"
        "- the dataset’s name\n"
        "- available datasets\n"
        "- column names or schema\n"
        "- specific data fields or codes\n\n"

        "DO NOT mention dataset structures, variables, column names, or technical terms. "
        "Questions should sound like what a business owner, planner, or citizen would ask, "
        "expecting AGORA to find and analyze relevant government data automatically.\n\n"

        "Each question should plausibly be answerable using this dataset (possibly combined "
        "with others), but without assuming the user knows that this dataset exists."
      )
    )


class DatasetQuestionGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.cot = dspy.ChainOfThought(GenerateQuestions)

    def forward(self, dataset_metadata: str, extracted_text_previews: str, retrieved_context: str, project_description: str):
        return self.cot(
            dataset_metadata=dataset_metadata,
            extracted_text_previews=extracted_text_previews,
            retrieved_context=retrieved_context,
            project_description=project_description,
        )


In [242]:
# ---- Put your AGORA project description text here (from the PDF) ----
# Key points: agentic open-data reasoning, feasibility questions (zoning/demographics/competition/etc.), web platform workflow. :contentReference[oaicite:5]{index=5}
AGORA_PROJECT_DESCRIPTION = """
The Agent-based Government Open-data Reasoning Assistant (AGORA) project focuses on
building an intelligent web platform where AI agents autonomously analyze open government
data to answer complex user queries. Instead of manually searching through government
portals, users can simply ask natural language questions—such as “What’s the feasibility of
opening a restaurant in Algiers?”—and receive comprehensive, actionable, data-driven
reports.
"""

# ---- This function is deprecated - because it uses top-k chunks which are related to the retireival query.
# ---- This is unnecassary.

# def generate_questions_for_dataset(dataset_id: str,
#                                   download_dir: str = "/content/dg_resources",
#                                   max_files: int = 6,
#                                   top_k_chunks: int = 8) -> str:
#     # 1) Fetch dataset JSON + URLs
#     ds = get_dataset_payload(dataset_id)
#     urls = extract_resource_urls(ds)

#     # 2) Download + extract text (bounded)
#     texts = []
#     for url in urls[:max_files]:
#         try:
#             path = download_file(url, download_dir)
#             text = extract_text_from_file(path)
#             texts.append(f"=== FILE: {os.path.basename(path)} ===\nSOURCE_URL: {url}\n\n{text}")
#         except Exception as e:
#             texts.append(f"=== FILE DOWNLOAD FAILED ===\nURL: {url}\nERROR: {e}")

#     # 3) Build chunks + retriever
#     all_chunks = []
#     for t in texts:
#         all_chunks.extend(chunk_text(t))

#     retriever = LocalTfidfRetriever(all_chunks if all_chunks else ["(no extracted content)"])

#     # 4) Retrieve most relevant chunks for “question generation”
#     #    (Query focuses on producing user-facing questions that AGORA would answer.)
#     retrieval_query = (
#         "Generate realistic user questions that this dataset could help answer within AGORA. "
#         "Focus on feasibility/analysis queries (demographics, zoning, competition, infrastructure, trends)."
#     )
#     top = retriever.topk(retrieval_query, k=top_k_chunks)
#     retrieved_context = "\n\n".join([f"[chunk #{i} | score={s:.3f}]\n{c}" for i, s, c in top])

#     # 5) Dataset metadata summary
#     dataset_metadata = summarize_dataset_metadata(ds)

#     # 6) DSPy generation
#     program = DatasetQuestionGenerator()
#     pred = program(
#         dataset_metadata=dataset_metadata,
#         retrieved_context=retrieved_context,
#         project_description=AGORA_PROJECT_DESCRIPTION,
#     )

#     return pred.questions


In [243]:
#dataset_id = "66178e5e022893ce023d5f04"  # <-- fill in a real data.gouv.fr dataset id
#questions = generate_questions_for_dataset(dataset_id)
#print(questions)


I am adding additional features now that let me see the DSPy history as well as how many tokens were used. It also verifies file downloads.

In [244]:
import logging

# More verbose internal logs (optional but useful)
logging.getLogger("dspy").setLevel(logging.DEBUG)  # DSPy FAQ mentions this approach. :contentReference[oaicite:1]{index=1}

# Track token usage (DSPy 2.6.16+)
dspy.configure(track_usage=True)  # usage can be read from Prediction.get_lm_usage(). :contentReference[oaicite:2]{index=2}


In [245]:
import os, hashlib
from typing import List, Dict, Any

def sha1_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def verify_downloads(downloaded_paths: List[str]) -> List[Dict[str, Any]]:
    report = []
    for p in downloaded_paths:
        item = {"path": p, "exists": os.path.exists(p)}
        if item["exists"]:
            item["bytes"] = os.path.getsize(p)
            item["sha1"] = sha1_file(p)
        report.append(item)
    return report

def print_download_report(report: List[Dict[str, Any]]):
    print("\nDOWNLOAD REPORT")
    print("=" * 80)
    for r in report:
        if not r["exists"]:
            print(f"❌ Missing: {r['path']}")
        else:
            print(f"✅ {r['path']} | {r['bytes']} bytes | sha1={r['sha1']}")


In [246]:
from typing import List
def print_text_previews(extracted_texts: List[str], max_chars: int = 800):
    print("\nEXTRACTED TEXT PREVIEWS (proof the program read the files)")
    print("=" * 80)
    for i, t in enumerate(extracted_texts, start=1):
        preview = (t[:max_chars] + "…") if len(t) > max_chars else t
        print(f"\n--- Extracted doc #{i} (chars={len(t)}) ---\n{preview}")

def string_text_previews(extracted_texts: List[str], max_chars: int = 800) -> str:
    lines = []
    lines.append("\nEXTRACTED TEXT PREVIEWS")
    lines.append("=" * 80)
    for i, t in enumerate(extracted_texts, start=1):
        preview = (t[:max_chars] + "…") if len(t) > max_chars else t
        lines.append(
            f"\n--- Extracted doc #{i} - First {len(t)} chars ---\n{preview}"
        )
    return "\n".join(lines)



In [247]:
import random

def generate_questions_for_dataset_debug(
    dataset_id: str,
    download_dir: str = "/content/dg_resources",
    max_files: int = 6,
    top_k_chunks: int = 8,
    show_history_n: int = 1,
    show_previews: bool = True
):
    ds = get_dataset_payload(dataset_id)
    urls = extract_resource_urls(ds)

    downloaded_paths = []
    extracted_texts = []

    # Download + extract
    for url in urls[:max_files]:
        try:
            path = download_file(url, download_dir)
            downloaded_paths.append(path)

            text = extract_text_from_file(path)
            extracted_texts.append(text)

            # vvv REMOVED because I felt it was unecessary.
            #extracted_texts.append(
            #    f"=== FILE: {os.path.basename(path)} ===\nSOURCE_URL: {url}\n\n{text}"
            #)
        except Exception as e:
            extracted_texts.append(f"=== FILE FAILED ===\nURL: {url}\nERROR: {e}")

    # Proof #1: downloaded files exist
    report = verify_downloads(downloaded_paths)
    print_download_report(report)

    # Proof #1.5: show some of what you extracted
    if show_previews:
        print_text_previews(extracted_texts, max_chars=800)

    # ADDED to show text previews (which usually includes headers)
    extracted_text_previews = string_text_previews(extracted_texts, max_chars=800)

    # vvvvv DELETED this retriever because I felt like it was unnecassary. New one is below it.
    # # Build retriever
    # all_chunks = []
    # for t in extracted_texts:
    #     all_chunks.extend(chunk_text(t))

    # retriever = LocalTfidfRetriever(all_chunks if all_chunks else ["(no extracted content)"])
    # retrieval_query = (
    #     "Generate realistic user questions that this dataset could help answer within AGORA. "
    #     "Focus on feasibility/analysis queries (demographics, zoning, competition, infrastructure, trends)."
    # )
    # top = retriever.topk(retrieval_query, k=top_k_chunks)
    # retrieved_context = "\n\n".join([f"[chunk #{i} | score={s:.3f}]\n{c}" for i, s, c in top])

    # Build chunks
    all_chunks = []
    for t in extracted_texts:
        all_chunks.extend(chunk_text(t))

    # Safety fallback
    if not all_chunks:
        all_chunks = ["(no extracted content)"]

    # Randomly sample chunks
    k = min(top_k_chunks, len(all_chunks))
    selected_chunks = random.sample(all_chunks, k)

    retrieved_context = "\n\n".join(
        f"[random chunk #{i}]\n{c}"
        for i, c in enumerate(selected_chunks)
    )

    print("\nRETRIEVED CONTEXT (what gets injected into the LLM call)")
    print("=" * 80)
    print(retrieved_context)

    dataset_metadata = summarize_dataset_metadata(ds)

    # Run DSPy program
    program = DatasetQuestionGenerator()
    pred = program(
        dataset_metadata=dataset_metadata,
        extracted_text_previews=extracted_text_previews,
        retrieved_context=retrieved_context,
        project_description=AGORA_PROJECT_DESCRIPTION,
    )

    # Proof #2: Inspect the exact prompt(s) DSPy sent to the LM
    # DSPy provides inspect_history utility for LLM invocations. :contentReference[oaicite:3]{index=3}
    if show_history_n and show_history_n > 0:
        print("\nDSPy LLM CALL HISTORY (recent)")
        print("=" * 80)
        dspy.inspect_history(n=show_history_n)

    # Optional: token usage (if supported by your DSPy version + enabled)
    try:
        usage = pred.get_lm_usage()
        print("\nLM USAGE")
        print("=" * 80)
        print(usage)
    except Exception:
        pass

    return {
        "questions": pred.questions,
        "downloaded_paths": downloaded_paths,
        "download_report": report,
        "extracted_texts": extracted_texts,
        "retrieved_context": retrieved_context,
        "dataset_metadata": dataset_metadata,
    }


The above code snippet is the MOST IMPORTANT in this entire notebook. This is where we are actually doing the whole process, everything up until this one is just helper functions.  Note that the returned things in this function include: questions, downloaded_paths, etc.

In the next code snippet we are testing out this function, which will also display the DSPy history - very useful for actually understanding what is going on

In [316]:
dataset_id = "53699630a3a729239d204918"  # fill this
out = generate_questions_for_dataset_debug(dataset_id, show_history_n=1)

print("\n\nGENERATED QUESTIONS")
print("=" * 80)
print(out["questions"])



DOWNLOAD REPORT

EXTRACTED TEXT PREVIEWS (proof the program read the files)

--- Extracted doc #1 (chars=443) ---
=== FILE FAILED ===
URL: http://catalogue.datalocale.fr/storage/f/2013-03-19T174639/ExportTrident20130319.zip
ERROR: HTTPConnectionPool(host='catalogue.datalocale.fr', port=80): Max retries exceeded with url: /storage/f/2013-03-19T174639/ExportTrident20130319.zip (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x7e77aa6f7aa0>: Failed to resolve 'catalogue.datalocale.fr' ([Errno -2] Name or service not known)"))

--- Extracted doc #2 (chars=437) ---
=== FILE FAILED ===
URL: http://catalogue.datalocale.fr/storage/f/2013-03-19T174734/ExportGTFS20130319.zip
ERROR: HTTPConnectionPool(host='catalogue.datalocale.fr', port=80): Max retries exceeded with url: /storage/f/2013-03-19T174734/ExportGTFS20130319.zip (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x7e77aa2cc680>: Failed to resolve 'catalogue.datalocale.fr' ([Errno

The next code snippet gives us additional information about our LLM calls using DSPy.

In [317]:
print("LM history length:", len(lm.history), "   --> (Note: this is probably number of times that the LLM was called IN TOTAL - not just in the last run)")
print("Last call keys:", lm.history[-1].keys())
print("Last call usage:", lm.history[-1].get("usage"))
print("Last call cost:", lm.history[-1].get("cost"))


LM history length: 19    --> (Note: this is probably number of times that the LLM was called IN TOTAL - not just in the last run)
Last call keys: dict_keys(['prompt', 'messages', 'kwargs', 'response', 'outputs', 'usage', 'cost', 'timestamp', 'uuid', 'model', 'response_model', 'model_type'])
Last call usage: {'completion_tokens': 1432, 'prompt_tokens': 1402, 'total_tokens': 2834, 'completion_tokens_details': CompletionTokensDetailsWrapper(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=704, rejected_prediction_tokens=0, text_tokens=None, image_tokens=None), 'prompt_tokens_details': PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None)}
Last call cost: 0.0032145


This is just to copy dataset summary:

In [318]:
ds = get_dataset_payload(dataset_id)
dataset_metadata = summarize_dataset_metadata(ds)
print(dataset_metadata)

ID: 53699630a3a729239d204918
Title: Horaires des lignes régulières du réseau transgironde
Description_short: None
Description: Les jeux de données mis à disposition permettent de disposer de la liste des arrêts d'autocar du réseau TransGironde avec pour chaque ligne du réseau, l'indication des points d'arrêts et des horaires théoriques de passage. 

Ces informations sont structurées aux formats Trident et GTFS dans un souci d'interopérabilité avec le format transmodel.
Organization: Conseil Départemental de la Gironde (slug=conseil-general-de-la-gironde, id=534fff67a3a7292c64a77d83)
License: odc-odbl
Frequency: quarterly
Created_at: 2013-09-18T08:45:32.480000+00:00
Last_update: 2013-09-18T08:45:32.480000+00:00
Tags: autobus, transgironde, transport
URI: https://www.data.gouv.fr/api/1/datasets/horaires-des-lignes-regulieres-du-reseau-transgironde-rdl/
Page: https://www.data.gouv.fr/datasets/horaires-des-lignes-regulieres-du-reseau-transgironde-rdl


This is just to copy the generated questions:

In [319]:
print(out["questions"])

1. Is there a direct regular bus connection between my town and the main city center, and how long does the ride usually take?
2. For a small retail shop I’m planning, which bus stops see the most frequent services during weekday shopping hours?
3. If I open a restaurant, what are the peak arrival and departure times at nearby bus stops so I can staff accordingly?
4. Can I rely on public buses for employees who start very early or finish very late shifts at my factory?
5. How convenient is public transport for students commuting from surrounding villages to the nearest secondary school?
6. Are there long time gaps in weekend service for certain towns that could hurt weekend tourism or local businesses?
7. For a one-day event, what are the first and last buses serving the nearest stops so I can plan event timings and shuttle needs?
8. Where are the best locations for a park-and-ride lot to capture commuters based on where buses run and how often?
9. Which rural towns have only a single 